# Pseudonymizer - hypothesis tests

Each hypothesis = one training run with a single change against the baseline, scored on the organizers' test set. Baseline = `BASELINE` (a run already on Drive, e.g. `wechsel_base`); the leaderboard (`runs/hypotheses.md`, with the history of runs) shows the difference to it.

| id | hypothesis | change |
|---|---|---|
| H1 | long context helps | `markussagen/xlm-roberta-longformer-base-4096`, window 4096 / stride 512 (full attention over 4096 - quadratic) |
| H2 | masks matter | baseline model, `--no-masks` (masked spans -> O) |
| H3 | oversampling НОМЕР matters | baseline model, `--oversample-number 0` |
| H4 | PII pre-training helps | `Wismut/nym-pii-multilingual` (mmBERT/ModernBERT, 40 PII types), window 1024 / stride 256, head replaced |

Runtime: **A100**. H1 ~1 h, H2/H3 ~15 min each.

In [ ]:
#@title 1. Code
REPO = "EnderLancer/anonimimizer"  #@param {type:"string"}
BRANCH = "dmelashych/datagen"  #@param {type:"string"}
import os
# private repo: add a GitHub token as Colab secret GH_TOKEN (left sidebar -> key icon -> enable notebook access)
try:
    from google.colab import userdata
    token = userdata.get('GH_TOKEN')
except Exception:
    token = None
url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
if not os.path.exists('/content/ai-camp/train'):
    !rm -rf /content/ai-camp && git clone -q --branch "$BRANCH" "$url" /content/ai-camp
else:
    !cd /content/ai-camp && git pull -q
assert os.path.exists('/content/ai-camp/train'), 'clone failed: check REPO/BRANCH and the GH_TOKEN secret'
%cd /content/ai-camp/train
!pip install -q -e . 2>&1 | tail -1
import os; os.environ.update(TRANSFORMERS_VERBOSITY='error', PYTHONWARNINGS='ignore', TOKENIZERS_PARALLELISM='false', HF_HUB_DISABLE_PROGRESS_BARS='1')
import torch
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'
print('GPU:', gpu)
if 'A100' not in gpu and 'L4' not in gpu:
    print('WARNING: not an A100 - Runtime -> Change runtime type -> A100 GPU (Colab Pro); on T4 halve every batch size')

In [ ]:
#@title 2. Data from Google Drive
from google.colab import drive
drive.mount('/content/drive')
import glob, os, shutil
from tqdm.auto import tqdm
DRIVE = '/content/drive/MyDrive/ai-camp'
os.makedirs('/content/ai-camp/data/silver', exist_ok=True); os.makedirs(f'{DRIVE}/runs', exist_ok=True)
src = f'{DRIVE}/silver' if os.path.exists(f'{DRIVE}/silver/train.parquet') else DRIVE
files = glob.glob(f'{src}/*.parquet') + glob.glob(f'{src}/*.jsonl')
assert files, f'no parquet/jsonl files found in {DRIVE} or {DRIVE}/silver'
for f in tqdm(files, desc='copy from Drive'):
    shutil.copy(f, '/content/ai-camp/data/silver/')
for need in ('train.parquet', 'dev.parquet', 'dev.jsonl'):
    assert os.path.exists(f'/content/ai-camp/data/silver/{need}'), f'{need} is missing in {src} - upload it to Drive first'
!ls -la /content/ai-camp/data/silver/
assert os.path.exists('/content/ai-camp/data/evaluate.py'), 'data/evaluate.py missing in the repo'
print('OK, data from', src)

In [ ]:
#@title 3. Hugging Face - every trained model is pushed to a PRIVATE repo <HF_USER>/pseudo-<run>
HF_USER = ""  #@param {type:"string"}  # your HF username or org; empty = do not push
if HF_USER:
    import os
    from google.colab import userdata
    from huggingface_hub import login
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')  # Colab: left sidebar -> Secrets -> HF_TOKEN (write access), enable notebook access
    login(os.environ['HF_TOKEN'])
else:
    print('HF_USER is empty: models are NOT pushed to the Hub')

In [ ]:
#@title 4. Hypotheses
BASELINE = "wechsel_base"  #@param {type:"string"}   # run name on Drive to compare against
# name: (model, extra CLI args, lr, batch, grad_accum, epochs)
HYPOTHESES = {
    'H1_longformer_4096': ('markussagen/xlm-roberta-longformer-base-4096', '--window 4096 --stride 512 --grad-checkpointing', 3e-5, 4, 8, 3),
    'H2_no_masks':        ('benjamin/roberta-base-wechsel-ukrainian', '--no-masks', 4e-5, 64, 1, 3),
    'H3_no_oversample':   ('benjamin/roberta-base-wechsel-ukrainian', '--oversample-number 0', 4e-5, 64, 1, 3),
    # PII-pretrained mmBERT (ModernBERT, 8k context, 40 PII types, 23 langs incl. uk): does PII pre-exposure beat a stronger Ukrainian encoder?
    'H4_nym_pii_mmbert':  ('Wismut/nym-pii-multilingual', '--window 1024 --stride 256', 3e-5, 32, 1, 3),
}
RUN_ONLY = ""  #@param {type:"string"}   # e.g. H1_longformer_4096 to run a single hypothesis
SKIP_DONE = True  #@param {type:"boolean"}
N_DOCS = 0  #@param {type:"integer"}
CPU_TIMING_DOCS = 20  #@param {type:"integer"}
TODO = {k: v for k, v in HYPOTHESES.items() if not RUN_ONLY or k == RUN_ONLY}
RUNS = [BASELINE] + list(HYPOTHESES); LEADERBOARD_NAME = 'hypotheses'
import os, shutil
if os.path.exists(f'{DRIVE}/runs/{BASELINE}') and not os.path.exists(f'runs/{BASELINE}'):
    shutil.copytree(f'{DRIVE}/runs/{BASELINE}', f'runs/{BASELINE}')
print(list(TODO))

In [ ]:
#@title Train all (skips a run whose model + hyper-parameters + data were already trained; saves to Drive right after training)
import os, shutil, time, json
items = list(TODO.items()); n = len(items); t_all = time.time()
for i, (run, (model, extra, lr, batch, accum, epochs)) in enumerate(items, 1):
    pct = int(100 * (i - 1) / n)
    print(f'\n[{i}/{n}  {pct}%] ===== {run}: {model} {extra} =====', flush=True)
    nd = f' --n-docs {N_DOCS}' if N_DOCS else ''
    nd += f' --push-to {HF_USER}/pseudo-{run}' if HF_USER else ''
    t0 = time.time()
    # --skip-if-done: train.py compares the fingerprint with the run on Drive and exits immediately if identical
    rc = os.system(f'python scripts/train.py --model {model} --out runs/{run} --epochs {epochs} --lr {lr} --batch {batch} '
                   f'--grad-accum {accum} --dev-docs 500 --skip-if-done {DRIVE}/runs/{run} {extra}{nd}')
    cfg = json.load(open(f'runs/{run}/config.json')) if os.path.exists(f'runs/{run}/config.json') else {}
    if rc != 0 or cfg.get('status') != 'done':
        print(f'\n[{i}/{n}] {run} FAILED (exit {rc}, status {cfg.get("status")}) - see runs/{run}/train.log; stopping the loop', flush=True)
        break
    !rm -rf runs/$run/checkpoints && mkdir -p $DRIVE/runs/$run && cp -r runs/$run/. $DRIVE/runs/$run/
    print(f'[{i}/{n}] {run}: fingerprint={cfg.get("fingerprint")} '
          f'({(time.time()-t0)/60:.1f} min; total {(time.time()-t_all)/60:.0f} min) -> saved to Drive', flush=True)
print(f'\n[{n}/{n}  100%] done in {(time.time()-t_all)/60:.0f} min')

In [ ]:
#@title CPU inference timing (Colab CPU is slower than a laptop; use it for ranking, not absolute numbers)
import json, os, subprocess, time
from tqdm.auto import tqdm
timing = {}
if CPU_TIMING_DOCS:
    !head -n $CPU_TIMING_DOCS ../data/test/court_cases.jsonl > /tmp/test_small.jsonl
    for run in tqdm([r for r in RUNS if os.path.exists(f'runs/{r}/model')], desc='cpu timing'):
        t0 = time.time()
        subprocess.run(['python','scripts/predict.py','--model',f'runs/{run}/model','--input','/tmp/test_small.jsonl','--output',f'/tmp/{run}_cpu.jsonl','--device','cpu','--threads','4','--batch','4'], capture_output=True)
        timing[run] = (time.time() - t0) / CPU_TIMING_DOCS
        print(f'{run:22s} {timing[run]:5.2f} s/doc -> test (240 docs) ~ {timing[run]*240/60:4.1f} min')
json.dump(timing, open(f'{DRIVE}/runs/cpu_timing_{LEADERBOARD_NAME}.json', 'w'))

In [ ]:
#@title Leaderboard on the organizers' test set + history of runs -> Drive
from IPython.display import Markdown, display
runs = ','.join(dict.fromkeys(RUNS))
baseline = f'--baseline {BASELINE}' if 'BASELINE' in globals() else ''
!python scripts/leaderboard.py --drive $DRIVE --name $LEADERBOARD_NAME --runs $runs $baseline
display(Markdown(open(f'{DRIVE}/runs/{LEADERBOARD_NAME}.md', encoding='utf-8').read()))